<a href="https://colab.research.google.com/github/BorysZhyhalo/goit-rdb-hw-05/blob/main/hw_5_rdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 5 — CTE, Recursive CTE and Materialized Views



## Task 1: Setup and Data Loading

In [4]:
!pip install -q pgserver==0.1.4 \
    --python-version 3.12 \
    --only-binary=:all: \
    --target=/usr/local/lib/python3.13/dist-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 10.4 MB/s eta 0:00:00


In [5]:
!pip install -q psycopg2-binary sqlalchemy pandas kagglehub

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 32.8 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [6]:
# Dataset import
import kagglehub

dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(dataset_dir)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
/kaggle/input/brazilian-ecommerce


In [7]:
import os

sorted(os.listdir(dataset_dir))

['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [8]:
files = [
    'olist_customers_dataset.csv',
    'olist_orders_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
]

for file_name in files:
    df_tmp = pd.read_csv(os.path.join(dataset_dir, file_name))
    print(f"\n{file_name}")
    print("shape:", df_tmp.shape)
    print("columns:", df_tmp.columns.tolist())


olist_customers_dataset.csv
shape: (99441, 5)
columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

olist_orders_dataset.csv
shape: (99441, 8)
columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

olist_order_items_dataset.csv
shape: (112650, 7)
columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

olist_products_dataset.csv
shape: (32951, 9)
columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

olist_sellers_dataset.csv
shape: (3095, 4)
columns: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


In [9]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)

    df.to_sql(
        table_name,
        engine,
        if_exists='replace',
        index=False,
        chunksize=10_000
    )

    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


### 1.2 Create Typed Tables with Constraints

In [10]:
with engine.begin() as conn:
    conn.execute(text("""

      DROP TABLE IF EXISTS olist_order_items CASCADE;
      DROP TABLE IF EXISTS olist_orders      CASCADE;
      DROP TABLE IF EXISTS olist_customers   CASCADE;
      DROP TABLE IF EXISTS olist_products    CASCADE;
      DROP TABLE IF EXISTS olist_sellers     CASCADE;

      CREATE TABLE olist_customers (
          customer_id              TEXT PRIMARY KEY,
          customer_unique_id       TEXT NOT NULL,
          customer_zip_code_prefix INTEGER,
          customer_city            TEXT,
          customer_state           CHAR(2) NOT NULL
      );

      CREATE TABLE olist_orders (
          order_id                      TEXT PRIMARY KEY,
          customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
          order_status                  TEXT NOT NULL,
          order_purchase_timestamp      TIMESTAMP NOT NULL,
          order_approved_at             TIMESTAMP,
          order_delivered_carrier_date  TIMESTAMP,
          order_delivered_customer_date TIMESTAMP,
          order_estimated_delivery_date TIMESTAMP,
          CHECK (order_status IN (
              'created', 'approved', 'invoiced', 'processing',
              'shipped', 'delivered', 'unavailable', 'canceled'
          ))
      );

      CREATE TABLE olist_products (
          product_id                 TEXT PRIMARY KEY,
          product_category_name      TEXT,
          product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
          product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
          product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
          product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
          product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
          product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
          product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
      );

      CREATE TABLE olist_sellers (
          seller_id              TEXT PRIMARY KEY,
          seller_zip_code_prefix INTEGER,
          seller_city            TEXT,
          seller_state           CHAR(2) NOT NULL
      );

      CREATE TABLE olist_order_items (
          order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
          order_item_id       INTEGER NOT NULL,
          product_id          TEXT REFERENCES olist_products(product_id),
          seller_id           TEXT REFERENCES olist_sellers(seller_id),
          shipping_limit_date TIMESTAMP,
          price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
          freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
          PRIMARY KEY (order_id, order_item_id)
      );

    """))
print("Successfully created typed tables.")

Successfully created typed tables.


### 1.3 Load Data from Raw to Typed Tables

In [11]:
with engine.begin() as conn:
    conn.execute(text("""

      INSERT INTO olist_customers
      SELECT
          customer_id,
          customer_unique_id,
          customer_zip_code_prefix::INTEGER,
          customer_city,
          customer_state::CHAR(2)
      FROM olist_customers_raw;

      INSERT INTO olist_orders
      SELECT
          order_id,
          customer_id,
          order_status,
          order_purchase_timestamp::TIMESTAMP,
          order_approved_at::TIMESTAMP,
          order_delivered_carrier_date::TIMESTAMP,
          order_delivered_customer_date::TIMESTAMP,
          order_estimated_delivery_date::TIMESTAMP
      FROM olist_orders_raw;

      INSERT INTO olist_products
      SELECT
          product_id,
          product_category_name,
          product_name_lenght::INTEGER,
          product_description_lenght::INTEGER,
          product_photos_qty::INTEGER,
          product_weight_g::INTEGER,
          product_length_cm::INTEGER,
          product_height_cm::INTEGER,
          product_width_cm::INTEGER
      FROM olist_products_raw;

      INSERT INTO olist_sellers
      SELECT
          seller_id,
          seller_zip_code_prefix::INTEGER,
          seller_city,
          seller_state::CHAR(2)
      FROM olist_sellers_raw;

      INSERT INTO olist_order_items
      SELECT
          order_id,
          order_item_id::INTEGER,
          product_id,
          seller_id,
          shipping_limit_date::TIMESTAMP,
          price::NUMERIC(12, 2),
          freight_value::NUMERIC(12, 2)
      FROM olist_order_items_raw;

  """))
print("Raw data successfully loaded into typed tables.")


Raw data successfully loaded into typed tables.


### 1.4 Validate Typed Table Row Counts

In [12]:
pd.read_sql("""
    SELECT 'customers' AS table_name, COUNT(*) AS n_rows
    FROM olist_customers

    UNION ALL

    SELECT 'orders', COUNT(*)
    FROM olist_orders

    UNION ALL

    SELECT 'order_items', COUNT(*)
    FROM olist_order_items

    UNION ALL

    SELECT 'products', COUNT(*)
    FROM olist_products

    UNION ALL

    SELECT 'sellers', COUNT(*)
    FROM olist_sellers

    ORDER BY table_name;
""", engine)

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,sellers,3095


## Task 2: CTE-chain feature pipeline

In [13]:
pd.read_sql("""

    WITH
    reference_date AS (
        -- Stage 0: stable reference date for historical dataset
        SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
        FROM olist_orders
    ),

    stage1_orders_total AS (
        -- Stage 1: orders + order_total + status
        SELECT
            o.customer_id,
            o.order_id,
            o.order_status,
            o.order_purchase_timestamp AS order_ts,
            o.order_purchase_timestamp::DATE AS order_date,
            COALESCE(SUM(
                COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
            ), 0)::NUMERIC(12, 2) AS order_total
        FROM olist_orders AS o
        LEFT JOIN olist_order_items AS oi
            ON oi.order_id = o.order_id
        GROUP BY
            o.customer_id,
            o.order_id,
            o.order_status,
            o.order_purchase_timestamp
    ),

    stage2_rfm AS (
      -- Stage 2: RFM features per customer
      SELECT
          s.customer_id,
          (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
          COUNT(*) AS frequency,
          SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
          MIN(s.order_date) AS first_order_date
      FROM stage1_orders_total AS s
      CROSS JOIN reference_date AS rd
      GROUP BY
          s.customer_id,
          rd.as_of_date
    ),

    stage3_cohort AS (
      -- Stage 3: cohort quarter from first order date
      SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
      FROM stage2_rfm AS r
    ),

    stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
      SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
    ),

    stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
      SELECT
          customer_id,
        AVG(CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
      FROM stage1_orders_total
      GROUP BY customer_id
    ),

stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)')
),

stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY
                    cr.category_revenue DESC,
                    cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)

SELECT
    c.customer_id,
    c.customer_state,

    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,

    coh.first_order_date,
    coh.cohort_quarter,

    COALESCE(
        aov.avg_order_value_30d,
        0
    )::NUMERIC(12, 2) AS avg_order_value_30d_recent,

    COALESCE(
        cancel.cancellation_rate,
        0
    )::NUMERIC(6, 4) AS cancellation_rate,

    COALESCE(
        pc.product_category_name,
        '(none)'
    ) AS preferred_category,

    COALESCE(
        pc.category_revenue,
        0
    )::NUMERIC(12, 2) AS preferred_category_revenue

FROM olist_customers AS c

LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id

LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1

LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id

LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id

ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 20;

""", engine)

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66
5,05455dfa7cd02f13d132aa7a6a9729c6,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0,agro_industria_e_comercio,6081.54
6,df55c14d1476a9a3467f131269c2477f,RJ,565,1,4950.34,2017-04-01,2017-04-01,4950.34,0.0,eletroportateis,4950.34
7,e0a2412720e9ea4f26c1ac985f6a7358,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0,cool_stuff,4809.44
8,24bbf5fd2f2e1b359ee7de94defc4a15,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0,eletroportateis,4764.34
9,3d979689f636322c62418b6346b1c6d2,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0,pcs,4681.78


## Task 3: Recursive CTE

### 3.1 Create synthetic referral graph

In [14]:
with engine.begin() as conn:
    conn.execute(text("""

    DROP TABLE IF EXISTS hw5_referrals CASCADE;

    CREATE TABLE hw5_referrals (
        referrer_customer_id TEXT NOT NULL
            REFERENCES olist_customers(customer_id),

        referred_customer_id TEXT NOT NULL
            REFERENCES olist_customers(customer_id),

        referred_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),

        PRIMARY KEY (referred_customer_id),

        CHECK (referrer_customer_id <> referred_customer_id)
    );

    WITH ordered_customers AS (
        SELECT
            customer_id,
            ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
        FROM olist_customers
        ORDER BY customer_id
        LIMIT 10
    )

    INSERT INTO hw5_referrals (
        referrer_customer_id,
        referred_customer_id
    )
    SELECT
        prev.customer_id AS referrer_customer_id,
        nxt.customer_id AS referred_customer_id
    FROM ordered_customers AS prev
    JOIN ordered_customers AS nxt
        ON nxt.rn = prev.rn + 1
    ON CONFLICT (referred_customer_id) DO NOTHING;

    SELECT *
    FROM hw5_referrals
    ORDER BY referrer_customer_id, referred_customer_id;

"""))

### 3.2 Recursive CTE

In [15]:
pd.read_sql("""

    WITH RECURSIVE referral_depth AS (
        -- ANCHOR: root customers, які не були referred іншими customers
        SELECT
            c.customer_id,
            1 AS depth,
            ARRAY[c.customer_id]::TEXT[] AS path_ids,
            c.customer_id::TEXT AS path_text
        FROM olist_customers AS c
        WHERE NOT EXISTS (
            SELECT 1
            FROM hw5_referrals AS r
            WHERE r.referred_customer_id = c.customer_id
        )

        UNION ALL

        -- RECURSIVE: перейти від referrer до його referred customers
        SELECT
            r.referred_customer_id AS customer_id,
            rd.depth + 1 AS depth,
            rd.path_ids || r.referred_customer_id AS path_ids,
            rd.path_text || ' -> ' || r.referred_customer_id AS path_text
        FROM hw5_referrals AS r
        JOIN referral_depth AS rd
            ON rd.customer_id = r.referrer_customer_id
        WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
          AND rd.depth < 50
    )

    SELECT
        depth,
        COUNT(*) AS n_customers,
        MAX(path_text) AS sample_path
    FROM referral_depth
    GROUP BY depth
    ORDER BY depth;

""", engine)

,depth,n_customers,sample_path
0,1,99432,ffffe8b65bbe3087b653a978c870db99
1,2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
2,3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
3,4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
4,5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
5,6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
6,7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
7,8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
8,9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
9,10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...


## Task 4: Materialized View

In [16]:
with engine.begin() as conn:
    conn.execute(text("""

      DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

      CREATE MATERIALIZED VIEW mv_olist_customer_features AS
      WITH
      reference_date AS (
          SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
          FROM olist_orders
      ),
      stage1_orders_total AS (
          SELECT
              o.customer_id,
              o.order_id,
              o.order_status,
              o.order_purchase_timestamp AS order_ts,
              o.order_purchase_timestamp::DATE AS order_date,
              COALESCE(SUM(
                  COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
              ), 0)::NUMERIC(12, 2) AS order_total
          FROM olist_orders AS o
          LEFT JOIN olist_order_items AS oi
              ON oi.order_id = o.order_id
          GROUP BY
              o.customer_id,
              o.order_id,
              o.order_status,
              o.order_purchase_timestamp
      ),
      stage2_rfm AS (
          SELECT
              s.customer_id,
              (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
              COUNT(*) AS frequency,
              SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
              MIN(s.order_date) AS first_order_date
          FROM stage1_orders_total AS s
          CROSS JOIN reference_date AS rd
          GROUP BY s.customer_id, rd.as_of_date
      ),
      stage3_cohort AS (
          SELECT
              r.*,
              DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
          FROM stage2_rfm AS r
      ),
      stage4_rolling_aov AS (
          SELECT
              s.customer_id,
              s.order_id,
              s.order_ts,
              s.order_total,
              AVG(s.order_total) OVER (
                  PARTITION BY s.customer_id
                  ORDER BY s.order_ts
                  RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
              )::NUMERIC(12, 2) AS avg_order_value_30d,
              ROW_NUMBER() OVER (
                  PARTITION BY s.customer_id
                  ORDER BY s.order_ts DESC, s.order_id DESC
              ) AS rn_recent
          FROM stage1_orders_total AS s
      ),
      stage5_cancel_rate AS (
          SELECT
              customer_id,
              AVG(
                  CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
              )::NUMERIC(6, 4) AS cancellation_rate
          FROM stage1_orders_total
          GROUP BY customer_id
      ),
      stage6_category_revenue AS (
          SELECT
              o.customer_id,
              COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
              SUM(
                  COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
              )::NUMERIC(12, 2) AS category_revenue
          FROM olist_orders AS o
          JOIN olist_order_items AS oi
              ON oi.order_id = o.order_id
          LEFT JOIN olist_products AS p
              ON p.product_id = oi.product_id
          GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
      ),
      stage7_preferred_category AS (
          SELECT
              customer_id,
              product_category_name,
              category_revenue
          FROM (
              SELECT
                  cr.*,
                  ROW_NUMBER() OVER (
                      PARTITION BY cr.customer_id
                      ORDER BY cr.category_revenue DESC, cr.product_category_name
                  ) AS rn
              FROM stage6_category_revenue AS cr
          ) AS ranked
          WHERE rn = 1
      )
      SELECT
          c.customer_id,
          c.customer_state,
          COALESCE(coh.recency_days, 9999) AS recency_days,
          COALESCE(coh.frequency, 0) AS frequency,
          COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
          coh.first_order_date,
          coh.cohort_quarter,
          COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
          COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
          COALESCE(pc.product_category_name, '(none)') AS preferred_category,
          COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
      FROM olist_customers AS c
      LEFT JOIN stage3_cohort AS coh
          ON coh.customer_id = c.customer_id
      LEFT JOIN stage4_rolling_aov AS aov
          ON aov.customer_id = c.customer_id
        AND aov.rn_recent = 1
      LEFT JOIN stage5_cancel_rate AS cancel
          ON cancel.customer_id = c.customer_id
      LEFT JOIN stage7_preferred_category AS pc
          ON pc.customer_id = c.customer_id;

      CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
          ON mv_olist_customer_features (customer_id);
"""))

In [17]:
pd.read_sql("""

    SELECT COUNT(*) AS n_rows
    FROM mv_olist_customer_features;

    SELECT *
    FROM mv_olist_customer_features
    ORDER BY monetary DESC NULLS LAST, customer_id
    LIMIT 5;

""", engine)

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66


In [18]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE UNIQUE INDEX IF NOT EXISTS idx_mv_olist_customer_features_customer_id
        ON mv_olist_customer_features (customer_id);
    """))

In [19]:
with engine.begin() as conn:
    conn.execute(text("""
        REFRESH MATERIALIZED VIEW CONCURRENTLY
        mv_olist_customer_features;
    """))

print("Concurrent refresh completed successfully.")

Concurrent refresh completed successfully.


In [20]:
pd.read_sql("""
    SELECT
        indexname,
        indexdef
    FROM pg_indexes
    WHERE tablename = 'mv_olist_customer_features';
""", engine)

,indexname,indexdef
0,idx_mv_olist_customer_features_customer_id,CREATE UNIQUE INDEX idx_mv_olist_customer_feat...


## Task 5: EXPLAIN (ANALYZE, BUFFERS).

### 5.1 Full pipeline

In [21]:
pd.read_sql("""

      EXPLAIN (ANALYZE, BUFFERS)
      WITH
      reference_date AS (
          SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
          FROM olist_orders
      ),

      stage1_orders_total AS (
          SELECT
              o.customer_id,
              o.order_id,
              o.order_status,
              o.order_purchase_timestamp AS order_ts,
              o.order_purchase_timestamp::DATE AS order_date,
              COALESCE(
                  SUM(
                      COALESCE(oi.price, 0)
                      + COALESCE(oi.freight_value, 0)
                  ),
                  0
              )::NUMERIC(12, 2) AS order_total
          FROM olist_orders AS o
          LEFT JOIN olist_order_items AS oi
              ON oi.order_id = o.order_id
          GROUP BY
              o.customer_id,
              o.order_id,
              o.order_status,
              o.order_purchase_timestamp
      ),

      stage2_rfm AS (
          SELECT
              s.customer_id,
              (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
              COUNT(*) AS frequency,
              SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
              MIN(s.order_date) AS first_order_date
          FROM stage1_orders_total AS s
          CROSS JOIN reference_date AS rd
          GROUP BY
              s.customer_id,
              rd.as_of_date
      )

      SELECT *
      FROM stage2_rfm
      ORDER BY monetary DESC NULLS LAST
      LIMIT 10;

""", engine)

,QUERY PLAN
0,Limit (cost=33552.08..33552.10 rows=10 width=...
1,"Buffers: shared hit=5966, temp read=3358 wri..."
2,-> Sort (cost=33552.08..33552.58 rows=200 ...
3,Sort Key: stage2_rfm.monetary DESC NUL...
4,Sort Method: top-N heapsort Memory: 27kB
5,"Buffers: shared hit=5966, temp read=33..."
6,-> Subquery Scan on stage2_rfm (cost...
7,"Buffers: shared hit=5966, temp r..."
8,-> HashAggregate (cost=33542.2...
9,"Group Key: o.customer_id, ..."


The full RFM pipeline took approximately 541.6 ms. The execution plan shows multiple join and aggregation operations, including a HashAggregate that used 21 batches and temporary disk I/O, indicating that customer-level aggregation is one of the more expensive parts of the computation. The final top-N sort used only 27 kB of memory, so sorting the final 10 rows was not a significant cost.

### 5.2 Lookup із Materialized View

In [22]:
pd.read_sql("""

      EXPLAIN (ANALYZE, BUFFERS)
      WITH target_customer AS (
          SELECT customer_id
          FROM olist_customers
          ORDER BY customer_id
          LIMIT 1
      )
      SELECT mv.*
      FROM mv_olist_customer_features AS mv
      JOIN target_customer AS t
          ON t.customer_id = mv.customer_id;

""",engine)

,QUERY PLAN
0,Nested Loop (cost=0.83..8.98 rows=1 width=158...
1,Buffers: shared hit=5 read=3
2,-> Limit (cost=0.42..0.55 rows=1 width=32)...
3,Buffers: shared hit=4
4,-> Index Only Scan using olist_custom...
5,Heap Fetches: 1
6,Buffers: shared hit=4
7,-> Index Scan using idx_mv_olist_customer_f...
8,Index Cond: (customer_id = olist_custo...
9,Buffers: shared hit=1 read=3


The lookup from the Materialized View completed in approximately 0.108 ms, compared with about 541.6 ms for the full RFM recomputation. The MV plan uses an Index Scan on the unique customer_id index and avoids the joins and aggregations required by the full pipeline. This demonstrates why materializing precomputed features can significantly reduce serving latency for customer-level ML features.

## Reflection

Based on the `EXPLAIN (ANALYZE, BUFFERS)` output, the most expensive part of the pipeline is the customer-level aggregation used to calculate the RFM features. The full recomputation took approximately 541.6 ms and included joins, `GroupAggregate`, and `HashAggregate` operations. The `HashAggregate` required 21 batches and temporary disk I/O, which indicates that aggregation consumed significant resources. In comparison, the final top-N sort used only 27 kB of memory, so sorting was not a major cost.

Caching these customer-level features in a Materialized View is useful when the same features are read frequently for analytics or ML inference. In this experiment, the indexed Materialized View lookup took only 0.108 ms and used an `Index Scan`, avoiding the joins and aggregations required by the full pipeline. For this historical e-commerce use case, I would choose a daily refresh cycle because customer features usually do not require second-by-second freshness.

A regular `VIEW` would be preferable when the underlying data changes frequently and consumers always require the latest values, especially if the query is relatively inexpensive.

The recursive CTE terminates using two protections. The `path_ids` array prevents visiting a customer that is already in the current path, while the `depth < 50` condition provides an additional depth guard. Therefore, a cycle such as `A - >B - > A` does not produce infinite recursion.

For this feature pipeline, the main trade-off is between freshness and query latency. The Materialized View significantly reduces serving latency and recomputation cost, but the cached features may become stale between refreshes. Storage cost is comparatively less important for this dataset.